In [0]:
%run ./00_config_and_logging

In [0]:
dbutils.widgets.text("load_type", "full", "Load type (full / incremental / backfill)")
dbutils.widgets.text("batch_id", "", "Batch ID (backfill only)")
dbutils.widgets.text("repo_path", "/Workspace/Users/l242547@lhr.nu.edu.pk/cyber-threat-intel", "Repo root path")

load_type = dbutils.widgets.get("load_type")
repo_path = dbutils.widgets.get("repo_path")
batch_id = None

if load_type == "full":
    data_base = f"{repo_path}/data/sample_raw/full_load"
    epss_file = "epss_full_sample.csv"
    kev_file = "kev_full_sample.json"
    nvd_file = "nvd_cves_full_sample.json"
    asset_file = "cortexa_assets_full_sample.csv"
else:
    incremental_base = f"{repo_path}/data/sample_raw/incremental"
    available = sorted([f.path.rstrip("/").split("/")[-1]
                        for f in dbutils.fs.ls(incremental_base) if f.isDir()])
    if not available:
        raise FileNotFoundError(f"No batch folders found under {incremental_base}")

    if load_type == "incremental":
        batch_id = available[-1]                      # latest batch
    elif load_type == "backfill":
        batch_id = dbutils.widgets.get("batch_id").strip()
        if batch_id not in available:
            raise ValueError(f"Batch '{batch_id}' not found. Available: {available}")
    else:
        raise ValueError("load_type must be full, incremental or backfill")

    data_base = f"{incremental_base}/{batch_id}"
    epss_file = f"epss_{batch_id}.csv"
    kev_file = f"kev_{batch_id}.json"
    nvd_file = f"nvd_{batch_id}.json"
    asset_file = f"cortexa_assets_{batch_id}.csv"     # doesn't exist yet, gets skipped + logged

print("Reading from:", data_base)
print("Batch:", batch_id if load_type != "full" else "full_load")

In [0]:
from datetime import datetime

def csv_header(path, comment=None):
    r = spark.read.option("header", "true")
    if comment:
        r = r.option("comment", comment)
    return r.csv(path).columns

def detect_drift(source_name, incoming_cols, expected_cols, run_id, load_type, target):
    added   = [c for c in incoming_cols if c not in expected_cols]
    missing = [c for c in expected_cols if c not in incoming_cols]
    if added or missing:
        msg = f"added={added} | missing={missing}"
        print(f"Schema change in {source_name}: {msg}")
        write_log(run_id, "Raw-to-Bronze", load_type, source_name, target,
                  datetime.now(), "Schema Change", error_message=msg)
    return added, missing

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col
from datetime import datetime

run_id = new_run_id()
start = datetime.now()

epss_path = f"{data_base}/{epss_file}"

try:
    incoming = csv_header(epss_path, "#")
    added, missing = detect_drift(epss_file, incoming, epss_schema.fieldNames(),
                                  run_id, load_type, BRONZE_EPSS)

    if not added and not missing:
        epss_df = spark.read.option("header", "true").option("comment", "#") \
            .schema(epss_schema).csv(epss_path)
    else:
        # read by column name, cast known cols, null-fill missing ones, keep new cols as strings
        raw = spark.read.option("header", "true").option("comment", "#").csv(epss_path)
        for f in epss_schema.fields:
            if f.name in raw.columns:
                raw = raw.withColumn(f.name, col(f.name).cast(f.dataType))
            else:
                raw = raw.withColumn(f.name, lit(None).cast(f.dataType))
        epss_df = raw

    epss_df = epss_df \
        .withColumn("load_timestamp", current_timestamp()) \
        .withColumn("source_file", lit(epss_file)) \
        .withColumn("load_type", lit(load_type))

    epss_df.write.format("delta").mode("append") \
        .option("mergeSchema", "true").saveAsTable(BRONZE_EPSS)

    write_log(run_id, "Raw-to-Bronze", load_type, epss_file, BRONZE_EPSS,
              start, "Success", rows_inserted=epss_df.count())
    print("EPSS rows written:", epss_df.count())
except Exception as e:
    write_log(run_id, "Raw-to-Bronze", load_type, epss_file, BRONZE_EPSS,
              start, "Failure", error_message=str(e))
    raise

In [0]:
from pyspark.sql.functions import explode, col, current_timestamp, lit
from datetime import datetime

run_id = new_run_id()
start = datetime.now()

kev_path = f"{data_base}/{kev_file}"

try:
    kev_raw_df = spark.read \
        .option("multiline", "true") \
        .schema(kev_root_schema) \
        .json(kev_path)

    kev_df = kev_raw_df.select(explode(col("vulnerabilities")).alias("vuln")) \
        .select("vuln.*")

    kev_df = kev_df \
        .withColumn("load_timestamp", current_timestamp()) \
        .withColumn("source_file", lit(kev_file)) \
        .withColumn("load_type", lit(load_type))

    kev_df.write.format("delta").mode("append").saveAsTable(BRONZE_KEV)

    write_log(run_id, "Raw-to-Bronze", load_type, kev_file, BRONZE_KEV,
              start, "Success", rows_inserted=kev_df.count())
    print("KEV rows written:", kev_df.count())
except Exception as e:
    if load_type != "full" and ("Path does not exist" in str(e) or "PATH_NOT_FOUND" in str(e) or "FileNotFound" in str(e)):
        print(f"KEV file not found in batch folder, skipping: {kev_file}")
        write_log(run_id, "Raw-to-Bronze", load_type, kev_file, BRONZE_KEV,
                  start, "Skipped", error_message="No KEV file in this batch")
    else:
        write_log(run_id, "Raw-to-Bronze", load_type, kev_file, BRONZE_KEV,
                  start, "Failure", error_message=str(e))
        raise

In [0]:
from pyspark.sql.functions import explode, col, current_timestamp, lit
from datetime import datetime

run_id = new_run_id()
start = datetime.now()

nvd_path = f"{data_base}/{nvd_file}"

try:
    nvd_raw_df = spark.read \
        .option("multiline", "true") \
        .schema(nvd_root_schema) \
        .json(nvd_path)

    nvd_df = nvd_raw_df.select(explode(col("vulnerabilities")).alias("item")) \
        .select("item.cve.*")

    nvd_df = nvd_df \
        .withColumn("load_timestamp", current_timestamp()) \
        .withColumn("source_file", lit(nvd_file)) \
        .withColumn("load_type", lit(load_type))

    nvd_df.write.format("delta").mode("append").saveAsTable(BRONZE_NVD)

    write_log(run_id, "Raw-to-Bronze", load_type, nvd_file, BRONZE_NVD,
              start, "Success", rows_inserted=nvd_df.count())
    print("NVD rows written:", nvd_df.count())
except Exception as e:
    if load_type != "full" and ("Path does not exist" in str(e) or "PATH_NOT_FOUND" in str(e) or "FileNotFound" in str(e)):
        print(f"NVD file not found in batch folder, skipping: {nvd_file}")
        write_log(run_id, "Raw-to-Bronze", load_type, nvd_file, BRONZE_NVD,
                  start, "Skipped", error_message="No NVD file in this batch")
    else:
        write_log(run_id, "Raw-to-Bronze", load_type, nvd_file, BRONZE_NVD,
                  start, "Failure", error_message=str(e))
        raise

In [0]:
run_id = new_run_id()
start = datetime.now()

asset_path = f"{data_base}/{asset_file}"
schema_to_use = asset_schema if load_type == "full" else asset_incremental_schema

try:
    asset_df = spark.read \
        .option("header", "true") \
        .schema(schema_to_use) \
        .csv(asset_path)

    asset_df = asset_df \
        .withColumn("load_timestamp", current_timestamp()) \
        .withColumn("source_file", lit(asset_file)) \
        .withColumn("load_type", lit(load_type))

    asset_df.write.format("delta").mode("append").saveAsTable(BRONZE_ASSETS)

    write_log(run_id, "Raw-to-Bronze", load_type, asset_file, BRONZE_ASSETS,
              start, "Success", rows_inserted=asset_df.count())
    print("Asset rows written:", asset_df.count())
except Exception as e:
    if load_type != "full" and "Path does not exist" in str(e):
        print(f"Asset file not found in batch folder, skipping: {asset_file}")
        write_log(run_id, "Raw-to-Bronze", load_type, asset_file, BRONZE_ASSETS,
                  start, "Skipped", error_message="No asset file in this batch")
    else:
        write_log(run_id, "Raw-to-Bronze", load_type, asset_file, BRONZE_ASSETS,
                  start, "Failure", error_message=str(e))
        raise

In [0]:
display(spark.table(LOG_TABLE))

In [0]:
# check count where cvid has substring less than 2021
display(spark.sql(f"select count(*) from {BRONZE_EPSS} where cve like 'CVE-2021%'"))
# spark.table(BRONZE_EPSS).schema

In [0]:
display(spark.table(LOG_TABLE).filter("load_type = 'backfill'").orderBy("start_time"))

In [0]:
display(spark.table(LOG_TABLE).filter("status = 'Schema Change'"))
spark.table(BRONZE_EPSS).printSchema()

In [0]:
display(spark.table(LOG_TABLE).orderBy("start_time"))